### Introduction
This notebook demonstrates how to use BigQuery SRA public dataset `nih-sra-datastore.sra` to get RNA-Seq gene counts and their analysis.   
This example use Google python clients instead of CLI tools.  
It shows possible metadata and analysis tables usage:
- Collecting gene counts from selecting run accessions by known run metadata attributes e.g. BioProject
- Flattering RNA-Seq specific attributes for subsequent filtering of obtained gene counts by feature counts QC metrics

### Google account and credentials
To run this tutorial you need:
- Google cloud project  
- Service account to use with appropriate permissions set, please visit [this NCBI page](https://www.ncbi.nlm.nih.gov/sra/docs/sra-bigquery/) for more information 
- Service account key certificate json file

### Source data location
**nih-sra-datastore.sra** is Google Cloud public BigQuery dataset containing 
- **metadata** [table schema](https://www.ncbi.nlm.nih.gov/sra/docs/sra-cloud-based-metadata-table/)
- **analysis** [table schema](https://www.ncbi.nlm.nih.gov/sra/docs/sra-cloud-based-examples/) TBD

### Setup
Assuming usage of Service Account credentials json file.  
Either set GOOGLE_APPLICATION_CREDENTIALS=path/to/you/credentials-file.json for you jyputer notebook session 
```
# e.g. 
export GOOGLE_APPLICATION_CREDENTIALS=path/to/you/credentials-file.json
.venv/bin/python3.13 .venv/bin/jupyter-notebook --no-browser --port=5000
```
Or uncomment and set it in the cell below

In [1]:
%%capture
%env PIP_DISABLE_PIP_VERSION_CHECK=1
#%env GOOGLE_APPLICATION_CREDENTIALS=path/to/you/credentials-file.json

In [2]:
!pip3 install -q google-cloud-bigquery[pandas] google-cloud-bigquery-storage

In [3]:
from google.cloud import bigquery
from google.oauth2 import service_account
import pandas as pd
import os
pd.set_option('display.max_colwidth', None)

### Obtainning BigQuery client

In [4]:
bq = bigquery.Client(credentials=service_account.Credentials.from_service_account_file(os.environ.get('GOOGLE_APPLICATION_CREDENTIALS')))

### Getting list of gene counts accessions and their locations by BioProject and optionally other metadata attributes

In [5]:
bioproject='PRJNA432002'

##### Either with pivoted analysis attributes in order to further narrow down by say alignment rate

In [6]:
dfa=bq.query(f"""
with 
X as (
  SELECT acc, target_run, item.k, item.v 
  FROM `nih-sra-datastore.sra.analysis`, unnest(attributes) as item
), 
A as (
  select * from X PIVOT(ANY_VALUE(v) FOR k IN ('total_reads', 'layout', 'intronic_reads', 'intergenic_reads', 'mito_count', 'ribo_count', 'decay_53_ratio', 'bases', 'bio_bases', 'strandness', 'unalaigned_reads', 'discordant_pairs', 'alignment_rate'))
),
Z as (  
  SELECT m.acc as run, 's3://sra-rnaseq-analysis/tsv/'||a.acc||'/'||m.acc||'.ncbi-gene-counts.tsv' path, a.* EXCEPT(target_run)
  FROM `nih-sra-datastore.sra.metadata` m left join A on a.target_run = m.acc
  where m.bioproject = '{bioproject}'
)
select * from Z
;
""").to_dataframe()

##### Available analysis QC metrics and attributes

In [7]:
dfa.head(1).T

,0
run,SRR6578603
path,s3://sra-rnaseq-analysis/tsv/SRZ1914938/SRR6578603.ncbi-gene-counts.tsv
acc,SRZ1914938
total_reads,633953
layout,PAIRED
intronic_reads,83596
intergenic_reads,14867
mito_count,7229
ribo_count,NaN
decay_53_ratio,0.0929


#### Note - it is possible not all runs will have gene counts available

In [9]:
print(dfa.path.isnull().sum())

9052


#### Or downloading analysis attributs as json for possible downstream storing and processing

In [10]:
dfb=bq.query(f"""
with 
A as (
    SELECT 
      acc, target_run, JSON_OBJECT(ARRAY_AGG(k), ARRAY_AGG(v)) AS json
    FROM 
      `nih-sra-datastore.sra.analysis`, UNNEST(attributes) AS item
    GROUP BY 
      acc, target_run 
), 
Z as (  
  SELECT m.acc as run, 's3://sra-rnaseq-analysis/tsv/'||A.acc||'/'||m.acc||'.ncbi-gene-counts.tsv' path, A.acc, json
  FROM `nih-sra-datastore.sra.metadata` m left join A on A.target_run = m.acc
  where m.bioproject = '{bioproject}'
)
select * from Z
;
""").to_dataframe()

In [12]:
df=dfb.head(1).T
df.style.set_properties(**{"text-align": "left"})

,0
run,SRR6578603
path,s3://sra-rnaseq-analysis/tsv/SRZ1914938/SRR6578603.ncbi-gene-counts.tsv
acc,SRZ1914938
json,"{""alignment_rate"":""89.59%"",""bases"":""126790600"",""bio_bases"":""126790600"",""decay_53_ratio"":""0.0929"",""discordant_pairs"":""4892"",""intergenic_reads"":""14867"",""intronic_reads"":""83596"",""layout"":""PAIRED"",""mito_count"":""7229"",""strandness"":""none"",""total_reads"":""633953"",""unaligned_reads"":""98790""}"
